In [17]:
import os 
os.chdir('/Users/hanli/JR/data-training/cafe_analytics_202511_ml')

In [18]:
#Config and connection testing
from google.cloud import bigquery
from src.config import settings

# For EDA, Data prepareation, Data visulisation
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#For network plot
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from itertools import combinations
from collections import Counter


In [19]:
#Create Base DF
client = bigquery.Client(project=settings.project_id)
df = client.query(f"""
SELECT distinct src.*,dp.product_name,dc.customer_id,dp.product_category from `jr-data-training.dbt_medallion_dev_gold.fact_order_items` src left join `jr-data-training.dbt_medallion_dev_gold.dim_products` dp on src.product_key = dp.product_key
left join `jr-data-training.dbt_medallion_dev_gold.dim_customers` dc on src.customer_key = dc.customer_key
where product_category is not null
""").to_dataframe()

In [22]:
(df['order_time'].astype(str))

0        08:43:16
1        10:28:52
2        11:58:43
3        11:31:21
4        08:24:14
           ...   
81481    06:43:26
81482    06:43:26
81483    09:29:54
81484    08:41:39
81485    09:45:43
Name: order_time, Length: 81486, dtype: object

In [11]:
df['order_time'] = pd.to_timedelta(df['order_time'].astype(str))

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81486 entries, 0 to 81485
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype          
---  ------               --------------  -----          
 0   order_id             81486 non-null  Int64          
 1   order_item_id        81486 non-null  Int64          
 2   order_date           81486 non-null  dbdate         
 3   order_time           81486 non-null  timedelta64[ns]
 4   order_total_price    81486 non-null  Int64          
 5   order_gst            81486 non-null  object         
 6   order_surcharge      81438 non-null  Int64          
 7   order_display_price  81450 non-null  object         
 8   order_display_gst    81450 non-null  object         
 9   cart_size            81486 non-null  object         
 10  cart_order_time      81486 non-null  object         
 11  customer_key         81486 non-null  object         
 12  product_key          81486 non-null  object         
 13  product_name    

In [23]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

def explore_data(df):
    """
    Comprehensive data exploration for recommendation system
    """
    print("="*80)
    print("DATA STRUCTURE ANALYSIS")
    print("="*80)
    
    print(f"\nDataset Shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
    print(f"\nData Types:\n{df.dtypes}")
    print(f"\nMissing Values:\n{df.isnull().sum()}")
    
    print("\n" + "="*80)
    print("CUSTOMER BEHAVIOR ANALYSIS")
    print("="*80)
    
    customer_stats = df.groupby('customer_key').agg({
        'order_id': 'nunique',
        'product_name': 'count',
        'order_total_price': 'sum'
    }).rename(columns={
        'order_id': 'total_orders',
        'product_name': 'total_items',
        'order_total_price': 'total_spent'
    })
    
    print(f"\nTotal Unique Customers: {df['customer_key'].nunique()}")
    print(f"Total Unique Products: {df['product_name'].nunique()}")
    print(f"Total Orders: {df['order_id'].nunique()}")
    
    print("\nCustomer Order Distribution:")
    print(customer_stats['total_orders'].describe())
    
    print("\nTop 10 Most Popular Products:")
    product_popularity = df['product_name'].value_counts().head(10)
    print(product_popularity)
    
    print("\n" + "="*80)
    print("TEMPORAL ANALYSIS")
    print("="*80)
    
    df['order_datetime'] = pd.to_datetime(df['order_time'].astype('str'))
    df['order_hour'] = df['order_datetime'].dt.hour
    df['order_dayofweek'] = df['order_datetime'].dt.dayofweek
    
    print("\nOrder Distribution by Hour:")
    print(df.groupby('order_hour')['order_id'].nunique().sort_values(ascending=False).head(5))
    
    print("\nOrder Distribution by Day of Week:")
    print(df.groupby('order_dayofweek')['order_id'].nunique())
    
    return df, customer_stats

df_explored, customer_stats = explore_data(df)

print("\n" + "="*80)
print("RECOMMENDATION SYSTEM REQUIREMENTS")
print("="*80)
print("\n1. Predict next order items based on customer history")
print("2. Suggest add-on items based on similar customer patterns")
print("\nKey Features Identified:")
print("  - Customer purchase history")
print("  - Product co-occurrence patterns")
print("  - Temporal patterns (time, day)")
print("  - Product categories")

DATA STRUCTURE ANALYSIS

Dataset Shape: (81486, 16)

Columns: ['order_id', 'order_item_id', 'order_date', 'order_time', 'order_total_price', 'order_gst', 'order_surcharge', 'order_display_price', 'order_display_gst', 'cart_size', 'cart_order_time', 'customer_key', 'product_key', 'product_name', 'customer_id', 'product_category']

Data Types:
order_id                Int64
order_item_id           Int64
order_date             dbdate
order_time             dbtime
order_total_price       Int64
order_gst              object
order_surcharge         Int64
order_display_price    object
order_display_gst      object
cart_size              object
cart_order_time        object
customer_key           object
product_key            object
product_name           object
customer_id             Int64
product_category       object
dtype: object

Missing Values:
order_id                0
order_item_id           0
order_date              0
order_time              0
order_total_price       0
order_gst      

In [31]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datetime import datetime, timedelta

class RecommendationDataPreparator:
    """
    Professional-grade data preparation for cafe recommendation system
    """
    
    def __init__(self, df):
        self.df = df.copy()
        self.customer_encoder = LabelEncoder()
        self.product_encoder = LabelEncoder()
        
    def prepare_datetime_features(self):
        """
        Create temporal features from order date and time
        """
        self.df['order_datetime'] = pd.to_datetime(
            self.df['order_date'].astype(str) + ' ' + self.df['order_time'].astype(str)
        )
        self.df = self.df.sort_values('order_datetime')
        
        self.df['order_hour'] = self.df['order_datetime'].dt.hour
        self.df['order_dayofweek'] = self.df['order_datetime'].dt.dayofweek
        self.df['order_month'] = self.df['order_datetime'].dt.month
        
        return self
    
    def create_customer_product_matrix(self):
        """
        Create customer-product interaction matrix
        """
        customer_product_counts = self.df.groupby(
            ['customer_key', 'product_name']
        ).size().reset_index(name='purchase_count')
        
        self.customer_product_matrix = customer_product_counts.pivot_table(
            index='customer_key',
            columns='product_name',
            values='purchase_count',
            fill_value=0
        )
        
        return self
    
    def create_order_sequences(self):
        """
        Create sequences of orders for each customer
        """
        order_sequences = []
        
        for customer in self.df['customer_key'].unique():
            customer_orders = self.df[
                self.df['customer_key'] == customer
            ].sort_values('order_datetime')
            
            orders_grouped = customer_orders.groupby('order_id').agg({
                'product_name': lambda x: list(x),
                'order_datetime': 'first',
                'order_total_price': 'first'
            }).sort_values('order_datetime')
            
            for idx in range(len(orders_grouped)):
                order_sequences.append({
                    'customer_key': customer,
                    'order_index': idx,
                    'order_datetime': orders_grouped.iloc[idx]['order_datetime'],
                    'products': orders_grouped.iloc[idx]['product_name'],
                    'order_total': orders_grouped.iloc[idx]['order_total_price']
                })
        
        self.order_sequences = pd.DataFrame(order_sequences)
        return self
    
    def create_training_data_for_next_item(self, min_orders=2):
        """
        Create training data for next-item prediction
        Target: predict products in current order based on previous orders
        """
        training_samples = []
        
        for customer in self.order_sequences['customer_key'].unique():
            customer_seq = self.order_sequences[
                self.order_sequences['customer_key'] == customer
            ].sort_values('order_datetime')
            
            if len(customer_seq) < min_orders:
                continue
            
            for idx in range(1, len(customer_seq)):
                prev_orders = customer_seq.iloc[:idx]
                current_order = customer_seq.iloc[idx]
                
                prev_products = []
                for products in prev_orders['products']:
                    prev_products.extend(products)
                
                prev_product_counts = pd.Series(prev_products).value_counts().to_dict()
                
                for target_product in current_order['products']:
                    training_samples.append({
                        'customer_key': customer,
                        'prev_product_counts': prev_product_counts,
                        'num_prev_orders': len(prev_orders),
                        'target_product': target_product,
                        'order_datetime': current_order['order_datetime']
                    })
        
        self.training_data = pd.DataFrame(training_samples)
        return self
    
    def create_feature_matrix(self):
        """
        Convert training data to feature matrix for ML models
        """
        all_products = sorted(self.df['product_name'].unique())
        
        feature_rows = []
        target_rows = []
        
        for idx, row in self.training_data.iterrows():
            features = []
            
            for product in all_products:
                features.append(row['prev_product_counts'].get(product, 0))
            
            features.append(row['num_prev_orders'])
            
            feature_rows.append(features)
            target_rows.append(row['target_product'])
        
        self.X = np.array(feature_rows)
        self.y = np.array(target_rows)
        self.feature_names = all_products + ['num_prev_orders']
        
        return self
    
    def temporal_train_test_split(self, test_size=0.2):
        """
        Split data based on time to prevent leakage
        """
        self.training_data = self.training_data.sort_values('order_datetime')

        split_idx = int(len(self.training_data) * (1 - test_size))
        split_date = self.training_data.iloc[split_idx]['order_datetime']

        train_mask = self.training_data['order_datetime'] < split_date
        test_mask = self.training_data['order_datetime'] >= split_date

        train_indices = self.training_data[train_mask].index
        test_indices = self.training_data[test_mask].index

        self.X_train = self.X[train_indices]
        self.X_test = self.X[test_indices]
        self.y_train = self.y[train_indices]
        self.y_test = self.y[test_indices]

        # --- FIX: remove unseen labels in test set ---
        train_labels = set(self.y_train)
        unseen = [i for i, label in enumerate(self.y_test) if label not in train_labels]

        if unseen:
            unseen_labels = set(self.y_test[unseen])
            print("\nWARNING: Dropping test rows with unseen labels:", unseen_labels)
            
            keep_idx = [i for i in range(len(self.y_test)) if i not in unseen]
            self.X_test = self.X_test[keep_idx]
            self.y_test = self.y_test[keep_idx]

        print(f"Training samples: {len(self.X_train)}")
        print(f"Test samples: {len(self.X_test)}")
        print(f"Split date: {split_date}")
        print(f"Feature dimensions: {self.X_train.shape}")

        return self

    
    def get_train_test_data(self):
        """
        Return prepared train-test data
        """
        return self.X_train, self.X_test, self.y_train, self.y_test


preparator = RecommendationDataPreparator(df)
preparator.prepare_datetime_features()
preparator.create_customer_product_matrix()
preparator.create_order_sequences()
preparator.create_training_data_for_next_item(min_orders=2)
preparator.create_feature_matrix()
preparator.temporal_train_test_split(test_size=0.2)

X_train, X_test, y_train, y_test = preparator.get_train_test_data()

print("\n" + "="*80)
print("DATA PREPARATION COMPLETED")
print("="*80)
print(f"\nUnique products in training: {len(np.unique(y_train))}")
print(f"Unique products in testing: {len(np.unique(y_test))}")
print(f"\nFeature names: {preparator.feature_names[:5]}... (showing first 5)")


Training samples: 61176
Test samples: 15282
Split date: 2023-05-20 12:18:38
Feature dimensions: (61176, 95)

DATA PREPARATION COMPLETED

Unique products in training: 91
Unique products in testing: 72

Feature names: ['(Entree) Pasta', '(Main) Pasta', '12-Hour Slow Cooked Pulled Pork', 'Acai Smoothie bowl (VO)', 'Affogato']... (showing first 5)


In [33]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    top_k_accuracy_score
)
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import time


class RecommendationModelTrainer:
    """
    Professional model training and evaluation for recommendation system
    """

    def __init__(self, X_train, X_test, y_train, y_test):
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test

        # Encode labels
        self.label_encoder = LabelEncoder()
        self.y_train_encoded = self.label_encoder.fit_transform(y_train)

        # Filter test labels to avoid unseen label failure
        valid_classes = set(self.label_encoder.classes_)
        mask = np.array([label in valid_classes for label in y_test])

        if not mask.all():
            removed = set(y_test[~mask])
            print("\nWARNING: Removed test samples with unseen labels:", removed)

        self.X_test = self.X_test[mask]
        self.y_test = self.y_test[mask]
        self.y_test_encoded = self.label_encoder.transform(self.y_test)

        self.models = {}
        self.results = {}


    # ----------------------------------------------------------------------
    # TRAINING FUNCTIONS
    # ----------------------------------------------------------------------

    def train_random_forest(self, n_estimators=100, max_depth=20, random_state=42):
        print("\n" + "="*80)
        print("TRAINING RANDOM FOREST CLASSIFIER")
        print("="*80)

        start = time.time()

        rf = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=10,
            min_samples_leaf=5,
            random_state=random_state,
            n_jobs=-1,
            class_weight='balanced'
        )

        rf.fit(self.X_train, self.y_train_encoded)
        self.models['random_forest'] = rf

        print(f"Training completed in {time.time() - start:.2f} seconds")
        return self


    def train_xgboost(self, n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42):
        print("\n" + "="*80)
        print("TRAINING XGBOOST CLASSIFIER")
        print("="*80)

        start = time.time()

        model = xgb.XGBClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            random_state=random_state,
            eval_metric='mlogloss',
            n_jobs=-1,
        )

        model.fit(self.X_train, self.y_train_encoded)
        self.models['xgboost'] = model

        print(f"Training completed in {time.time() - start:.2f} seconds")
        return self


    def train_logistic_regression(self, max_iter=1000, random_state=42):
        print("\n" + "="*80)
        print("TRAINING LOGISTIC REGRESSION")
        print("="*80)

        start = time.time()

        lr = LogisticRegression(
            max_iter=max_iter,
            random_state=random_state,
            n_jobs=-1,
            class_weight='balanced',
            multi_class='multinomial'
        )

        lr.fit(self.X_train, self.y_train_encoded)
        self.models['logistic_regression'] = lr

        print(f"Training completed in {time.time() - start:.2f} seconds")
        return self


    # ----------------------------------------------------------------------
    # EVALUATION FUNCTION (FIXED)
    # ----------------------------------------------------------------------

    def evaluate_model(self, model_name, top_k=5):
        print("\n" + "="*80)
        print(f"EVALUATING {model_name.upper()}")
        print("="*80)

        model = self.models[model_name]

        # Predictions
        y_pred = model.predict(self.X_test)
        y_proba = model.predict_proba(self.X_test)

        # Ensure sklearn knows full class space
        full_class_indices = np.arange(len(self.label_encoder.classes_))

        # Metrics
        accuracy = accuracy_score(self.y_test_encoded, y_pred)

        top_k_acc = top_k_accuracy_score(
            y_true=self.y_test_encoded,
            y_score=y_proba,
            k=top_k,
            labels=full_class_indices
        )

        precision, recall, f1, _ = precision_recall_fscore_support(
            self.y_test_encoded,
            y_pred,
            average='weighted',
            zero_division=0
        )

        # Save results
        self.results[model_name] = {
            'accuracy': accuracy,
            f'top_{top_k}_accuracy': top_k_acc,
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        }

        # Print report
        print(f"\nAccuracy: {accuracy:.4f}")
        print(f"Top-{top_k} Accuracy: {top_k_acc:.4f}")
        print(f"Precision (weighted): {precision:.4f}")
        print(f"Recall (weighted): {recall:.4f}")
        print(f"F1 Score (weighted): {f1:.4f}")

        # Top prediction distribution
        print("\nTop 10 Predicted Classes:")
        unique, counts = np.unique(y_pred, return_counts=True)
        top_pred = sorted(zip(unique, counts), key=lambda x: x[1], reverse=True)[:10]

        for idx, count in top_pred:
            name = self.label_encoder.inverse_transform([idx])[0]
            print(f"  {name}: {count}")

        return self


    # ----------------------------------------------------------------------
    # TOP-K PRODUCT RECOMMENDATIONS
    # ----------------------------------------------------------------------

    def get_top_k_predictions(self, model_name, sample_idx=0, k=5):
        model = self.models[model_name]
        sample = self.X_test[sample_idx:sample_idx+1]
        proba = model.predict_proba(sample)[0]

        top_idx = np.argsort(proba)[-k:][::-1]

        return [
            {
                "product": self.label_encoder.inverse_transform([i])[0],
                "confidence": proba[i]
            }
            for i in top_idx
        ]


    # ----------------------------------------------------------------------
    # MODEL COMPARISON
    # ----------------------------------------------------------------------

    def compare_models(self):
        print("\n" + "="*80)
        print("MODEL COMPARISON")
        print("="*80)

        df = pd.DataFrame(self.results).T
        df = df.sort_values("accuracy", ascending=False)

        print(df)

        best = df.index[0]
        print("\n" + "*"*80)
        print(f"BEST MODEL: {best.upper()}")
        print("*"*80)

        return best, df


In [34]:
trainer = RecommendationModelTrainer(X_train, X_test, y_train, y_test)

trainer.train_random_forest()
trainer.evaluate_model("random_forest")

trainer.train_xgboost()
trainer.evaluate_model("xgboost")

trainer.train_logistic_regression()
trainer.evaluate_model("logistic_regression")

best_model, results = trainer.compare_models()



TRAINING RANDOM FOREST CLASSIFIER
Training completed in 0.98 seconds

EVALUATING RANDOM_FOREST

Accuracy: 0.3360
Top-5 Accuracy: 0.7878
Precision (weighted): 0.5808
Recall (weighted): 0.3360
F1 Score (weighted): 0.3914

Top 10 Predicted Classes:
  Cappuccino: 1950
  Latte: 1569
  Flat White: 1011
  Hash Brown: 957
  Babychino: 781
  Toasted Roll: 662
  Orange Almond Cake (GF,VG): 580
  Ice Latte: 539
  Cookies: 486
  Date and Walnut Balls (GF,VG): 453

TRAINING XGBOOST CLASSIFIER
Training completed in 29.24 seconds

EVALUATING XGBOOST

Accuracy: 0.5616
Top-5 Accuracy: 0.8753
Precision (weighted): 0.5042
Recall (weighted): 0.5616
F1 Score (weighted): 0.5183

Top 10 Predicted Classes:
  Latte: 5908
  Cappuccino: 3627
  Flat White: 1468
  Toasted Roll: 1351
  Toastie: 607
  Babychino: 466
  Muffin: 365
  Hot Chocolate: 302
  Piccolo: 212
  Mocha: 142

TRAINING LOGISTIC REGRESSION
Training completed in 108.08 seconds

EVALUATING LOGISTIC_REGRESSION

Accuracy: 0.1698
Top-5 Accuracy: 0.5308

In [36]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

class AddonRecommender:
    """
    Market Basket Analysis for add-on item recommendations
    """
    
    def __init__(self, df):
        self.df = df.copy()
        self.association_rules_df = None
        self.item_basket = None
        
    def prepare_transaction_data(self):
        """
        Prepare transaction-level data for market basket analysis
        """
        print("="*80)
        print("PREPARING TRANSACTION DATA FOR MARKET BASKET ANALYSIS")
        print("="*80)
        
        transactions = self.df.groupby('order_id')['product_name'].apply(list).values
        
        self.transactions = list(transactions)
        
        print(f"\nTotal transactions: {len(self.transactions)}")
        print(f"Sample transaction: {self.transactions[0]}")
        
        te = TransactionEncoder()
        te_ary = te.fit(self.transactions).transform(self.transactions)
        self.item_basket = pd.DataFrame(te_ary, columns=te.columns_)
        
        print(f"\nTransaction matrix shape: {self.item_basket.shape}")
        print(f"Unique items: {self.item_basket.shape[1]}")
        
        return self
    
    def generate_frequent_itemsets(self, min_support=0.01):
        """
        Generate frequent itemsets using Apriori algorithm
        """
        print("\n" + "="*80)
        print("GENERATING FREQUENT ITEMSETS")
        print("="*80)
        
        frequent_itemsets = apriori(
            self.item_basket, 
            min_support=min_support, 
            use_colnames=True
        )
        
        frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))
        
        self.frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False)
        
        print(f"\nTotal frequent itemsets found: {len(self.frequent_itemsets)}")
        print(f"Minimum support threshold: {min_support}")
        
        print("\nTop 10 Most Frequent Itemsets:")
        print(self.frequent_itemsets.head(10).to_string())
        
        return self
    
    def generate_association_rules(self, metric='lift', min_threshold=1.0):
        """
        Generate association rules from frequent itemsets
        """
        print("\n" + "="*80)
        print("GENERATING ASSOCIATION RULES")
        print("="*80)
        
        rules = association_rules(
            self.frequent_itemsets, 
            metric=metric, 
            min_threshold=min_threshold
        )
        
        rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
        rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))
        
        self.association_rules_df = rules.sort_values('lift', ascending=False)
        
        print(f"\nTotal association rules found: {len(self.association_rules_df)}")
        print(f"Metric: {metric}")
        print(f"Minimum threshold: {min_threshold}")
        
        return self
    
    def get_addon_recommendations(self, current_items, top_n=5, min_confidence=0.3):
        """
        Get add-on recommendations for current order items
        """
        if self.association_rules_df is None:
            raise ValueError("Association rules not generated. Run generate_association_rules() first.")
        
        current_items_set = set(current_items)
        
        matching_rules = self.association_rules_df[
            self.association_rules_df['antecedents'].apply(
                lambda x: x.issubset(current_items_set)
            )
        ]
        
        matching_rules = matching_rules[matching_rules['confidence'] >= min_confidence]
        
        matching_rules = matching_rules[
            ~matching_rules['consequents'].apply(
                lambda x: x.issubset(current_items_set)
            )
        ]
        
        recommendations = []
        seen_products = set()
        
        for _, rule in matching_rules.iterrows():
            for product in rule['consequents']:
                if product not in seen_products and product not in current_items_set:
                    recommendations.append({
                        'product': product,
                        'confidence': rule['confidence'],
                        'lift': rule['lift'],
                        'support': rule['support'],
                        'based_on': list(rule['antecedents'])
                    })
                    seen_products.add(product)
                    
                    if len(recommendations) >= top_n:
                        break
            
            if len(recommendations) >= top_n:
                break
        
        return recommendations
    
    def display_rule_statistics(self):
        """
        Display comprehensive statistics about association rules
        """
        print("\n" + "="*80)
        print("ASSOCIATION RULES STATISTICS")
        print("="*80)
        
        print("\nConfidence Distribution:")
        print(self.association_rules_df['confidence'].describe())
        
        print("\nLift Distribution:")
        print(self.association_rules_df['lift'].describe())
        
        print("\nSupport Distribution:")
        print(self.association_rules_df['support'].describe())
        
        print("\nTop 10 Rules by Lift:")
        top_rules = self.association_rules_df.nlargest(10, 'lift')[
            ['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift']
        ]
        print(top_rules.to_string())
        
        return self


addon_recommender = AddonRecommender(df)
addon_recommender.prepare_transaction_data()
addon_recommender.generate_frequent_itemsets(min_support=0.01)
addon_recommender.generate_association_rules(metric='lift', min_threshold=1.0)
addon_recommender.display_rule_statistics()

print("\n" + "="*80)
print("EXAMPLE ADD-ON RECOMMENDATIONS")
print("="*80)

example_orders = [
    ['Latte'],
    ['Latte', 'Croissant'],
    ['Cappuccino', 'Muffin']
]

for order in example_orders:
    print(f"\nCurrent Order: {order}")
    recommendations = addon_recommender.get_addon_recommendations(
        order, 
        top_n=5, 
        min_confidence=0.3
    )
    
    if recommendations:
        print("Recommended Add-ons:")
        for i, rec in enumerate(recommendations, 1):
            print(f"  {i}. {rec['product']}")
            print(f"     Confidence: {rec['confidence']:.3f} | Lift: {rec['lift']:.3f}")
            print(f"     Based on: {rec['based_on']}")
    else:
        print("No recommendations found with current thresholds")

ModuleNotFoundError: No module named 'mlxtend'